In [1]:
!pip install -q transformers datasets huggingface_hub soundfile librosa scipy numpy tqdm
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118

In [2]:
from huggingface_hub import login
login()  # Enter your HF token when prompted

import os, json, random, torch, torchaudio, soundfile as sf, numpy as np
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

OUTPUT_DIR = Path("/content/tts_output")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "train_set").mkdir(exist_ok=True)
(OUTPUT_DIR / "test_set").mkdir(exist_ok=True)
(OUTPUT_DIR / "generated_eval").mkdir(exist_ok=True)
(OUTPUT_DIR / "reference_voices").mkdir(exist_ok=True)
print("Output folders created.")

Using device: cuda
Output folders created.


In [3]:
from datasets import load_dataset
from collections import defaultdict

MALE_SPEAKER_ID   = "S4257113700379470"
FEMALE_SPEAKER_ID = "S4258579700363788"
TARGET_SPEAKERS   = {MALE_SPEAKER_ID, FEMALE_SPEAKER_ID}
MAX_PER_SPEAKER   = 60
id_col            = "speaker_id"
text_col          = "text"

print(f"Streaming config='Bengali', looking for 2 speakers...")
print(f"  Male:   {MALE_SPEAKER_ID}")
print(f"  Female: {FEMALE_SPEAKER_ID}")

stream = load_dataset(
    "ai4bharat/indicvoices_r",
    "Bengali",
    split="train",
    streaming=True,
)

collected = defaultdict(list)
checked   = 0

for sample in stream:
    checked += 1
    sid = sample.get(id_col, "")

    if sid in TARGET_SPEAKERS and len(collected[sid]) < MAX_PER_SPEAKER:
        collected[sid].append(sample)
        print(f"  [{sid}] collected #{len(collected[sid])} at sample {checked}")

    if checked % 10000 == 0:
        counts = {k[-8:]: len(v) for k, v in collected.items()}
        print(f"  Scanned {checked:,} samples | collected: {counts}")

    if all(len(collected[s]) >= MAX_PER_SPEAKER for s in TARGET_SPEAKERS):
        print(f"Both speakers complete after scanning {checked:,} samples.")
        break

male_samples   = collected[MALE_SPEAKER_ID]
female_samples = collected[FEMALE_SPEAKER_ID]

print(f"\nMale   ({MALE_SPEAKER_ID}): {len(male_samples)} samples")
print(f"Female ({FEMALE_SPEAKER_ID}): {len(female_samples)} samples")

Streaming config='Bengali', looking for 2 speakers...
  Male:   S4257113700379470
  Female: S4258579700363788


Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/155 [00:00<?, ?it/s]

  [S4258579700363788] collected #1 at sample 1
  [S4257113700379470] collected #1 at sample 2
  [S4257113700379470] collected #2 at sample 14
  [S4257113700379470] collected #3 at sample 15
  [S4257113700379470] collected #4 at sample 16
  [S4257113700379470] collected #5 at sample 17
  [S4258579700363788] collected #2 at sample 103
  [S4258579700363788] collected #3 at sample 104
  [S4258579700363788] collected #4 at sample 105
  [S4258579700363788] collected #5 at sample 106
  [S4258579700363788] collected #6 at sample 107
  [S4258579700363788] collected #7 at sample 108
  [S4258579700363788] collected #8 at sample 109
  [S4258579700363788] collected #9 at sample 449
  [S4258579700363788] collected #10 at sample 450
  [S4258579700363788] collected #11 at sample 451
  [S4258579700363788] collected #12 at sample 452
  [S4258579700363788] collected #13 at sample 453
  [S4258579700363788] collected #14 at sample 1238
  [S4258579700363788] collected #15 at sample 1239
  [S4258579700363788

In [4]:
import random, csv

random.seed(42)

def split_speaker_data(samples, speaker_label, test_ratio=0.2):
    indices = list(range(len(samples)))
    random.shuffle(indices)
    n_test    = max(1, int(len(indices) * test_ratio))
    test_idx  = indices[:n_test]
    train_idx = indices[n_test:]
    train = [samples[i] for i in train_idx]
    test  = [samples[i] for i in test_idx]
    print(f"[{speaker_label}] Train: {len(train)} | Test (unseen): {len(test)}")
    return train, test

male_train,   male_test   = split_speaker_data(male_samples,   "MALE")
female_train, female_test = split_speaker_data(female_samples, "FEMALE")

def save_metadata(split_data, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["index", "speaker_id", "text"])
        for i, row in enumerate(split_data):
            writer.writerow([i, row[id_col], row[text_col]])

save_metadata(male_train,   OUTPUT_DIR / "train_set" / "male_train_meta.csv")
save_metadata(male_test,    OUTPUT_DIR / "test_set"  / "male_test_meta.csv")
save_metadata(female_train, OUTPUT_DIR / "train_set" / "female_train_meta.csv")
save_metadata(female_test,  OUTPUT_DIR / "test_set"  / "female_test_meta.csv")
print("Metadata CSVs saved.")

[MALE] Train: 48 | Test (unseen): 12
[FEMALE] Train: 48 | Test (unseen): 12
Metadata CSVs saved.


In [5]:
import soundfile as sf
import numpy as np
from tqdm import tqdm

def extract_audio(sample):
    audio = sample["audio"]
    if hasattr(audio, "get_all_samples"):
        result = audio.get_all_samples()
        arr = np.array(result.data, dtype=np.float32)
        sr  = int(result.sample_rate)
    elif isinstance(audio, dict):
        arr = np.array(audio["array"], dtype=np.float32)
        sr  = int(audio["sampling_rate"])
    else:
        import torchaudio
        arr_t, sr = torchaudio.load(audio)
        arr = arr_t.numpy().astype(np.float32)

    # Fix shape: (channels, samples) -> 1D mono
    if arr.ndim == 2:
        arr = arr.mean(axis=0)
    arr = arr.squeeze()

    # Normalize if values outside [-1, 1]
    max_val = np.abs(arr).max()
    if max_val > 1.0:
        arr = arr / 32768.0

    return arr, sr

# Quick test before saving all
print("Testing extract_audio on first male test sample...")
arr0, sr0 = extract_audio(male_test[0])
print(f"  shape: {arr0.shape}, dtype: {arr0.dtype}, sr: {sr0}, min: {arr0.min():.3f}, max: {arr0.max():.3f}")

def save_ground_truth(test_data, gender, max_save=20):
    out = OUTPUT_DIR / "test_set" / f"{gender}_ground_truth"
    out.mkdir(exist_ok=True)
    for i, sample in enumerate(tqdm(test_data[:max_save], desc=f"Saving {gender} GT")):
        arr, sr = extract_audio(sample)
        sf.write(str(out / f"{gender}_gt_{i:03d}.wav"), arr, sr)
    print(f"[{gender}] Ground truth saved → {out}")

save_ground_truth(male_test,   "male")
save_ground_truth(female_test, "female")

Testing extract_audio on first male test sample...
  shape: (247392,), dtype: float32, sr: 48000, min: -0.918, max: 0.973


Saving male GT: 100%|██████████| 12/12 [00:00<00:00, 22.25it/s]


[male] Ground truth saved → /content/tts_output/test_set/male_ground_truth


Saving female GT: 100%|██████████| 12/12 [00:00<00:00, 46.14it/s]

[female] Ground truth saved → /content/tts_output/test_set/female_ground_truth


In [6]:
def get_reference_audio(samples, gender, n_ref=5):
    clips, sr_out = [], 16000
    for i, sample in enumerate(samples[:n_ref]):
        arr, sr = extract_audio(sample)
        sr_out = sr
        clips.append(arr)
    ref_audio = np.concatenate(clips)
    ref_path  = OUTPUT_DIR / "reference_voices" / f"{gender}_reference.wav"
    sf.write(str(ref_path), ref_audio, sr_out)
    print(f"[{gender}] Reference saved → {ref_path} | SR: {sr_out} | Duration: {len(ref_audio)/sr_out:.1f}s")
    return ref_path, sr_out

male_ref_path,   male_sr   = get_reference_audio(male_train,   "male")
female_ref_path, female_sr = get_reference_audio(female_train, "female")

[male] Reference saved → /content/tts_output/reference_voices/male_reference.wav | SR: 48000 | Duration: 40.1s
[female] Reference saved → /content/tts_output/reference_voices/female_reference.wav | SR: 48000 | Duration: 33.6s


In [7]:
import json

EVAL_JSON_PATH = "bengali_evaluation_set.json"  # upload this file to Colab first

with open(EVAL_JSON_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# Key confirmed as "bengali_sentence"
sentences = [
    {
        "text"    : entry["bengali_sentence"],
        "id"      : entry["id"],
        "category": entry.get("benchmark_category", "")
    }
    for entry in eval_data[:20]
]

print(f"Loaded {len(sentences)} evaluation sentences.")
for i, s in enumerate(sentences[:3]):
    print(f"  [{i}] {s['text'][:80]}")

Loaded 20 evaluation sentences.
  [0] অন্ধকার ঘরে কাঁপা হাতে সে পুরোনো পুঁথি আর রথের ভাঙা চাকা খুঁজছে।
  [1] ফাল্গুনের মেলায় ভণ্ড সাধুর কথায় ঘণ্টা বাজতেই এক অদ্ভুত কম্পন তৈরি হলো।
  [2] শহরের এই উচ্চ অট্টালিকার ছাদে দাঁড়ালে উদ্দাম বাতাসের শব্দে অন্য সত্তার খোঁজ মেলে


In [10]:
# Cell 8
import os, sys

# Step 1: Clone with submodules
if not os.path.exists("/content/CosyVoice3"):
    !git clone --recursive https://github.com/kawshikbuet17/CosyVoice3-TTS-Bengali-Finetuning.git /content/CosyVoice3
    %cd /content/CosyVoice3
    !git submodule update --init --recursive
else:
    %cd /content/CosyVoice3
    print("Repo already cloned.")

# Step 2: Fix onnxruntime — CPU version only
!pip uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null || true
!pip install -q onnxruntime==1.18.0

# Step 3: Install exact transformers + numpy CosyVoice3 needs
!pip install -q "transformers==4.51.3" "numpy==1.26.4"

# Step 4: Reinstall huggingface-hub to a version that works for BOTH
# datasets (gated) and CosyVoice3
!pip install -q "huggingface-hub>=0.23.0,<1.0.0"

# Step 5: Install Matcha-TTS submodule
!pip install -q /content/CosyVoice3/third_party/Matcha-TTS --no-deps

# Step 6: Core deps
!pip install -q modelscope
!pip install -q openai-whisper
!pip install -q conformer hydra-core omegaconf
!pip install -q librosa soundfile inflect
!pip install -q WeTextProcessing --no-deps || true

# Step 7: Add cosyvoice to Python path directly (no pip install -e needed)
sys.path.insert(0, "/content/CosyVoice3")
sys.path.insert(0, "/content/CosyVoice3/third_party/Matcha-TTS")

# Step 8: Verify import works
try:
    from cosyvoice.cli.cosyvoice import AutoModel
    from cosyvoice.utils.common import set_all_random_seed
    print("CosyVoice imports OK.")
except Exception as e:
    print(f"Import error: {e}")

print("Installation complete.")

/content/CosyVoice3
Repo already cloned.
Found existing installation: onnxruntime 1.18.0
Uninstalling onnxruntime-1.18.0:
  Successfully uninstalled onnxruntime-1.18.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
espnet 202511 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
dtw-python 1.7.5 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_versi

In [12]:
# Final fix: correct huggingface-hub version that satisfies all packages
!pip install -q "huggingface-hub==0.30.2"
!pip install -q onnxruntime==1.18.0
!pip install -q hyperpyyaml

import sys
sys.path.insert(0, "/content/CosyVoice3")
sys.path.insert(0, "/content/CosyVoice3/third_party/Matcha-TTS")

# Now test imports
try:
    from cosyvoice.cli.cosyvoice import AutoModel
    from cosyvoice.utils.common import set_all_random_seed
    import torchaudio, torch, numpy as np, soundfile as sf
    from pathlib import Path
    print("ALL IMPORTS OK — ready for Cell 9")
except ImportError as e:
    print(f"Import error: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 32.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.38.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.30.2 which is incompatible.
ALL IMPORTS OK — ready for Cell 9


In [ ]:
# Install all remaining dependencies at once
!pip install -q wget x-transformers einops vector-quantize-pytorch vocos encodec

# Reload model
import sys
sys.path.insert(0, "/content/CosyVoice3")
sys.path.insert(0, "/content/CosyVoice3/third_party/Matcha-TTS")
%cd /content/CosyVoice3

from cosyvoice.cli.cosyvoice import AutoModel
from cosyvoice.utils.common import set_all_random_seed
import torchaudio, torch, numpy as np, soundfile as sf
from pathlib import Path

MODEL_LOCAL_DIR = "/content/CosyVoice3/pretrained_models/Fun-CosyVoice3-0.5B"

print("Loading model...")
cosyvoice = AutoModel(model_dir=MODEL_LOCAL_DIR)
SAMPLE_RATE = cosyvoice.sample_rate
print(f"Model loaded. Sample rate: {SAMPLE_RATE}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 92.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.3/120.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.6 MB/s eta 0:00:00
/content/CosyVoice3
Loading model...


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:69: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


In [2]:
# Cell 9: Set paths, download model, load
import sys
sys.path.insert(0, "/content/CosyVoice3")
sys.path.insert(0, "/content/CosyVoice3/third_party/Matcha-TTS")
%cd /content/CosyVoice3

import torchaudio, torch, numpy as np, soundfile as sf
from pathlib import Path
import os

os.makedirs("/content/CosyVoice3/pretrained_models/Fun-CosyVoice3-0.5B", exist_ok=True)
MODEL_LOCAL_DIR = "/content/CosyVoice3/pretrained_models/Fun-CosyVoice3-0.5B"

# Check if already downloaded
already_downloaded = os.path.exists(os.path.join(MODEL_LOCAL_DIR, "cosyvoice3.yaml"))

if not already_downloaded:
    print("Downloading model weights (~7GB, ~4 min)...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        "kawshikbuet17/bengali-cosyvoice3-tts",
        local_dir=MODEL_LOCAL_DIR,
        repo_type="model",
        ignore_patterns=["*.incomplete"],
    )
    print("Download complete.")
else:
    print("Model weights already downloaded, skipping.")

# Load model
from cosyvoice.cli.cosyvoice import AutoModel
from cosyvoice.utils.common import set_all_random_seed

print("Loading model...")
cosyvoice = AutoModel(model_dir=MODEL_LOCAL_DIR)
SAMPLE_RATE = cosyvoice.sample_rate
print(f"Model loaded. Sample rate: {SAMPLE_RATE}")

/content/CosyVoice3


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

CosyVoice-BlankEN/model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

speech_tokenizer_v3.onnx:   0%|          | 0.00/969M [00:00<?, ?B/s]

Download complete.


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading model...


ErrorDuringImport: problem in cosyvoice.flow.flow_matching - ModuleNotFoundError: No module named 'wget'

In [1]:
# Cell 10: Restore variables from earlier cells
# (only needed if kernel restarted — otherwise skip)
from pathlib import Path
import soundfile as sf
import numpy as np

OUTPUT_DIR = Path("/content/tts_output")
MALE_SPEAKER_ID   = "S4257113700379470"
FEMALE_SPEAKER_ID = "S4258579700363788"
id_col   = "speaker_id"
text_col = "text"

male_ref_path   = OUTPUT_DIR / "reference_voices" / "male_reference.wav"
female_ref_path = OUTPUT_DIR / "reference_voices" / "female_reference.wav"

# Reload eval sentences
import json
with open("bengali_evaluation_set.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)
sentences = [{"text": e["bengali_sentence"], "id": e["id"]} for e in eval_data[:20]]
print(f"Eval sentences loaded: {len(sentences)}")
print(f"Male ref exists: {male_ref_path.exists()}")
print(f"Female ref exists: {female_ref_path.exists()}")

Eval sentences loaded: 20
Male ref exists: True
Female ref exists: True


In [4]:
# Reload cosyvoice model (run this after any kernel restart)
import sys
sys.path.insert(0, "/content/CosyVoice3")
sys.path.insert(0, "/content/CosyVoice3/third_party/Matcha-TTS")
%cd /content/CosyVoice3

from cosyvoice.cli.cosyvoice import AutoModel
from cosyvoice.utils.common import set_all_random_seed

MODEL_LOCAL_DIR = "/content/CosyVoice3/pretrained_models/Fun-CosyVoice3-0.5B"

print("Loading model (weights already on disk)...")
cosyvoice = AutoModel(model_dir=MODEL_LOCAL_DIR)
SAMPLE_RATE = cosyvoice.sample_rate
print(f"Model loaded. Sample rate: {SAMPLE_RATE}")

/content/CosyVoice3


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading model (weights already on disk)...


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:69: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Model loaded. Sample rate: 24000


In [12]:
# Cell 11 (FIXED): Trim reference audio to max 25s + generate male eval
import sys
sys.path.insert(0, "/content/CosyVoice3")
sys.path.insert(0, "/content/CosyVoice3/third_party/Matcha-TTS")
%cd /content/CosyVoice3

import torchaudio, torch, numpy as np, soundfile as sf, json
from tqdm import tqdm
from pathlib import Path
from cosyvoice.utils.common import set_all_random_seed

OUTPUT_DIR      = Path("/content/tts_output")
male_ref_path   = OUTPUT_DIR / "reference_voices" / "male_reference.wav"
female_ref_path = OUTPUT_DIR / "reference_voices" / "female_reference.wav"
SAMPLE_RATE     = cosyvoice.sample_rate
PROMPT_SR       = 16000
MAX_REF_SECS    = 25   # stay safely under 30s limit
COSYVOICE3_PREFIX = "You are a helpful assistant.<|endofprompt|>"

def prepare_ref_audio(ref_wav_path, max_secs=MAX_REF_SECS):
    """Resample to 16kHz mono, trim to max_secs, save temp file."""
    waveform, sr = torchaudio.load(str(ref_wav_path))
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != PROMPT_SR:
        waveform = torchaudio.functional.resample(waveform, sr, PROMPT_SR)
    # Trim to max_secs
    max_samples = int(max_secs * PROMPT_SR)
    if waveform.shape[1] > max_samples:
        waveform = waveform[:, :max_samples]
        print(f"  Trimmed to {max_secs}s")
    temp_path = str(ref_wav_path).replace(".wav", "_16k.wav")
    torchaudio.save(temp_path, waveform, PROMPT_SR)
    duration = waveform.shape[1] / PROMPT_SR
    print(f"  Ref ready: {duration:.1f}s @ {PROMPT_SR}Hz → {temp_path}")
    return temp_path

def generate_speech(text, ref_audio_path, seed=42, speed=1.0):
    set_all_random_seed(seed)
    tts_text = COSYVOICE3_PREFIX + text.strip()
    chunks = []
    for item in cosyvoice.inference_cross_lingual(
        tts_text, ref_audio_path, stream=False, speed=float(speed)
    ):
        chunks.append(item["tts_speech"].detach().cpu().numpy().flatten())
    if not chunks:
        raise RuntimeError("No audio generated.")
    return np.concatenate(chunks)

# Prepare trimmed references
print("Preparing reference audios (trimmed to 25s)...")
male_ref_16k   = prepare_ref_audio(male_ref_path)
female_ref_16k = prepare_ref_audio(female_ref_path)

# Load eval sentences
with open("/content/bengali_evaluation_set.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)
sentences = [{"text": e["bengali_sentence"], "id": e["id"]} for e in eval_data[:20]]
print(f"Loaded {len(sentences)} eval sentences.")

# Generate MALE eval set
male_gen_dir = OUTPUT_DIR / "generated_eval" / "male"
male_gen_dir.mkdir(parents=True, exist_ok=True)

print(f"\nGenerating {len(sentences)} sentences — MALE voice...")
for i, entry in enumerate(tqdm(sentences, desc="Male TTS")):
    try:
        wav = generate_speech(entry["text"], male_ref_16k)
        sf.write(str(male_gen_dir / f"male_eval_{i:03d}.wav"), wav, SAMPLE_RATE)
        print(f"  [{i:02d}] ✓ {entry['text'][:60]}")
    except Exception as e:
        print(f"  [{i:02d}] ✗ {e}")

print(f"\nMale eval done → {male_gen_dir}")

/content/CosyVoice3
Preparing reference audios (trimmed to 25s)...
  Trimmed to 25s
  Ref ready: 25.0s @ 16000Hz → /content/tts_output/reference_voices/male_reference_16k.wav
  Trimmed to 25s
  Ref ready: 25.0s @ 16000Hz → /content/tts_output/reference_voices/female_reference_16k.wav
Loaded 20 eval sentences.

Generating 20 sentences — MALE voice...


  0%|          | 0/1 [00:00<?, ?it/s]/content/CosyVoice3/cosyvoice/cli/model.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with self.llm_context, torch.cuda.amp.autocast(self.fp16 is True and hasattr(self.llm, 'vllm') is False):
/content/CosyVoice3/cosyvoice/cli/model.py:426: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.fp16):

Male TTS:   5%|▌         | 1/20 [00:17<05:26, 17.17s/it]

  [00] ✓ অন্ধকার ঘরে কাঁপা হাতে সে পুরোনো পুঁথি আর রথের ভাঙা চাকা খুঁ



Male TTS:  10%|█         | 2/20 [00:31<04:40, 15.57s/it]

  [01] ✓ ফাল্গুনের মেলায় ভণ্ড সাধুর কথায় ঘণ্টা বাজতেই এক অদ্ভুত কম্পন


...

  [17] ✓ ব্যাঙ্কের ম্যানেজার গ্যারাজে গিয়ে ট্যাক্সি ভাড়া করে এয়ারপ



Male TTS:  95%|█████████▌| 19/20 [04:52<00:15, 15.16s/it]

  [18] ✓ হঠাৎ উৎসবের দিনে নদীর জল এতটাই বেড়ে গেল যে গ্রামের সবাই খুব



Male TTS: 100%|██████████| 20/20 [05:09<00:00, 15.45s/it]

  [19] ✓ নৌকো করে নদীর থৈথৈ জলে ভাসার সময়, উনুনের মিষ্টি ধোঁয়ায় তার

Male eval done → /content/tts_output/generated_eval/male


In [13]:
# Cell 12: Generate female eval set
female_gen_dir = OUTPUT_DIR / "generated_eval" / "female"
female_gen_dir.mkdir(parents=True, exist_ok=True)

print(f"Generating {len(sentences)} sentences — FEMALE voice...")
for i, entry in enumerate(tqdm(sentences, desc="Female TTS")):
    try:
        wav = generate_speech(entry["text"], female_ref_16k)
        sf.write(str(female_gen_dir / f"female_eval_{i:03d}.wav"), wav, SAMPLE_RATE)
        print(f"  [{i:02d}] ✓ {entry['text'][:60]}")
    except Exception as e:
        print(f"  [{i:02d}] ✗ {e}")

print(f"Female eval done → {female_gen_dir}")

Generating 20 sentences — FEMALE voice...


Female TTS:   5%|▌         | 1/20 [00:15<04:56, 15.61s/it]

  [00] ✓ অন্ধকার ঘরে কাঁপা হাতে সে পুরোনো পুঁথি আর রথের ভাঙা চাকা খুঁ



Female TTS:  10%|█         | 2/20 [00:31<04:40, 15.56s/it]

  [01] ✓ ফাল্গুনের মেলায় ভণ্ড সাধুর কথায় ঘণ্টা বাজতেই এক অদ্ভুত কম্পন


...

  [17] ✓ ব্যাঙ্কের ম্যানেজার গ্যারাজে গিয়ে ট্যাক্সি ভাড়া করে এয়ারপ



Female TTS:  95%|█████████▌| 19/20 [04:52<00:15, 15.24s/it]

  [18] ✓ হঠাৎ উৎসবের দিনে নদীর জল এতটাই বেড়ে গেল যে গ্রামের সবাই খুব



Female TTS: 100%|██████████| 20/20 [05:07<00:00, 15.38s/it]

  [19] ✓ নৌকো করে নদীর থৈথৈ জলে ভাসার সময়, উনুনের মিষ্টি ধোঁয়ায় তার
Female eval done → /content/tts_output/generated_eval/female


In [14]:
# Restore dataset variables (run after kernel restart)
from datasets import load_dataset
from collections import defaultdict
from huggingface_hub import HfFolder
import random, sys
from pathlib import Path

# Paths
OUTPUT_DIR      = Path("/content/tts_output")
male_ref_path   = OUTPUT_DIR / "reference_voices" / "male_reference.wav"
female_ref_path = OUTPUT_DIR / "reference_voices" / "female_reference.wav"

# Restore sys path
sys.path.insert(0, "/content/CosyVoice3")
sys.path.insert(0, "/content/CosyVoice3/third_party/Matcha-TTS")

# Settings
MALE_SPEAKER_ID   = "S4257113700379470"
FEMALE_SPEAKER_ID = "S4258579700363788"
TARGET_SPEAKERS   = {MALE_SPEAKER_ID, FEMALE_SPEAKER_ID}
MAX_PER_SPEAKER   = 60
id_col            = "speaker_id"
text_col          = "text"
HF_TOKEN          = HfFolder.get_token()
PROMPT_SR         = 16000

# Re-stream dataset to get samples
print("Re-streaming dataset to restore speaker samples...")
stream = load_dataset(
    "ai4bharat/indicvoices_r",
    "Bengali",
    split="train",
    streaming=True,
    token=HF_TOKEN,
)

collected = defaultdict(list)
checked   = 0
for sample in stream:
    checked += 1
    sid = sample.get(id_col, "")
    if sid in TARGET_SPEAKERS and len(collected[sid]) < MAX_PER_SPEAKER:
        collected[sid].append(sample)
    if checked % 10000 == 0:
        counts = {k[-8:]: len(v) for k, v in collected.items()}
        print(f"  Scanned {checked:,} | {counts}")
    if all(len(collected[s]) >= MAX_PER_SPEAKER for s in TARGET_SPEAKERS):
        print(f"Done after {checked:,} samples.")
        break

male_samples   = collected[MALE_SPEAKER_ID]
female_samples = collected[FEMALE_SPEAKER_ID]

# Recreate train/test split with same seed
random.seed(42)

def split_speaker_data(samples, test_ratio=0.2):
    indices = list(range(len(samples)))
    random.shuffle(indices)
    n_test    = max(1, int(len(indices) * test_ratio))
    train = [samples[i] for i in indices[n_test:]]
    test  = [samples[i] for i in indices[:n_test]]
    return train, test

male_train,   male_test   = split_speaker_data(male_samples)
female_train, female_test = split_speaker_data(female_samples)

print(f"Male   — Train: {len(male_train)} | Test: {len(male_test)}")
print(f"Female — Train: {len(female_train)} | Test: {len(female_test)}")

# Restore ref audio paths (16k versions)
male_ref_16k   = str(male_ref_path).replace(".wav", "_16k.wav")
female_ref_16k = str(female_ref_path).replace(".wav", "_16k.wav")

print(f"Male   ref 16k exists: {Path(male_ref_16k).exists()}")
print(f"Female ref 16k exists: {Path(female_ref_16k).exists()}")
print("All variables restored. Ready for Cell 13.")

Re-streaming dataset to restore speaker samples...


Resolving data files:   0%|          | 0/246 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/155 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [15]:
# Cell 13: Generate TTS on unseen test set for objective evaluation
for gender, test_split, ref_16k in [
    ("male",   male_test,   male_ref_16k),
    ("female", female_test, female_ref_16k),
]:
    gen_dir = OUTPUT_DIR / "test_set" / f"{gender}_generated"
    gen_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[{gender}] Generating {min(len(test_split), 20)} test samples...")
    for i, sample in enumerate(tqdm(test_split[:20], desc=f"{gender} test")):
        try:
            wav = generate_speech(sample[text_col], ref_16k)
            sf.write(str(gen_dir / f"{gender}_test_gen_{i:03d}.wav"), wav, SAMPLE_RATE)
        except Exception as e:
            print(f"  [{i}] ✗ {e}")
    print(f"  Saved → {gen_dir}")


[male] Generating 12 test samples...


  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/content/CosyVoice3/cosyvoice/cli/model.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with self.llm_context, torch.cuda.amp.autocast(self.fp16 is True and hasattr(self.llm, 'vllm') is False):
/content/CosyVoice3/cosyvoice/cli/model.py:426: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.fp16):

male test: 100%|██████████| 12/12 [04:19<00:00, 21.59s/it]


  Saved → /content/tts_output/test_set/male_generated

[female] Generating 12 test samples...


female test: 100%|██████████| 12/12 [03:36<00:00, 18.04s/it]

  Saved → /content/tts_output/test_set/female_generated


In [18]:
# Cell 14: Zip all outputs and download
import zipfile
from google.colab import files

zip_path = "/content/tts_output.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fpath in OUTPUT_DIR.rglob("*"):
        if fpath.is_file():
            zf.write(fpath, fpath.relative_to(OUTPUT_DIR))

print(f"Zipped → {zip_path}")
files.download(zip_path)

Zipped → /content/tts_output.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>